#### MILE -- mit qwen_posterior_utils (_predict-Fix + geteiltes Modul)


In [ ]:
from pathlib import Path

# ============================================================
# CONFIG
# MILE FULL-WARMUP STABILITY TEST
# ============================================================

MASTER_DIR = Path(
    "/dss/dsshome1/00/ra58vit2/Masterarbeit"
)

QWEN_REPO = MASTER_DIR / "bayes_sub_inf"
MILE_REPO = MASTER_DIR / "MILE"

BASELINE_SCRIPT = (
    QWEN_REPO
    / "experiments"
    / "ag_news_qwen_lora"
    / "evaluate_saved_baseline.py"
)

RESULT_DIR = (
    MASTER_DIR
    / "method_results"
    / "mile_qwen_agnews"
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 2
POSTERIOR_KEY_SEED = 2027

# ------------------------------------------------------------
# Posterior data
# ------------------------------------------------------------

N_PER_CLASS = 32         # 32 x 4 = 128 examples
SEQ_LEN = 32

# ------------------------------------------------------------
# Prior
# ------------------------------------------------------------

# War 1.0 (Standard-Default aus dem MILE-Paper). Bei 540672 Dimensionen
# ist das massiv zu breit: eine typische Stichprobe aus N(0, 1.0^2 * I)
# hat eine Norm von ~sqrt(540672)*1.0 ~= 735 -- weit weg von der
# tatsaechlichen MAP-Norm (~13.0). Das MILE-Paper selbst skaliert die
# Prior-Varianz fuer groessere Modelle runter (0.1-0.4 statt 1.0 fuer
# ihre CNN/ATT-Modelle, deutlich kleiner als unser 540k-dim LoRA-Raum).
# Empirisch hergeleitet aus der MAP-Norm: 13.0148 / sqrt(540672) ~= 0.0177.
PRIOR_STD = 0.02

# ------------------------------------------------------------
# MCLMC
# ------------------------------------------------------------

N_CHAINS = 1

# NOCH NICHT GETESTET (letzte Session abgebrochen) -- konservativere
# Adaption, um die Divergenz aus dem letzten Warmup-Lauf zu vermeiden
# (Original: TRUST_IN_ESTIMATE=1.5, DESIRED_ENERGY_VAR_START=0.0005,
# DESIRED_ENERGY_VAR_END=0.0001, WARMUP_STEPS=100).
WARMUP_STEPS = 150
N_SAMPLES = 10
N_THINNING = 1

DIAGONAL_PRECONDITIONING = True

DESIRED_ENERGY_VAR_START = 0.0001
DESIRED_ENERGY_VAR_END = 0.00002

TRUST_IN_ESTIMATE = 0.3

NUM_EFFECTIVE_SAMPLES = 10

STEP_SIZE_INIT = 1e-5

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

EVAL_BATCH_SIZE = 64

# False (Default)  -> auswerten auf dem 128er-Posterior-Subset
#                      (schneller Dev-/Stabilitaets-Check).
# True              -> auswerten auf dem echten AG-News-Testset
#                      (7600 Beispiele, fuer die finalen Thesis-Zahlen).
EVAL_ON_TEST_SET = False

# ------------------------------------------------------------
# Automatic run name
# ------------------------------------------------------------

N_POSTERIOR_CONFIG = 4 * N_PER_CLASS

RUN_NAME = (
    f"mile_balanced{N_POSTERIOR_CONFIG}"
    f"_seq{SEQ_LEN}"
    f"_prior{PRIOR_STD}"
    f"_warmup{WARMUP_STEPS}"
    f"_samples{N_SAMPLES}"
    f"_ess{NUM_EFFECTIVE_SAMPLES}"
)

# ------------------------------------------------------------
# Print configuration
# ------------------------------------------------------------

print("======================================")
print("MILE WARMUP STABILITY TEST")
print("======================================")
print("Run name:          ", RUN_NAME)
print("Examples/class:    ", N_PER_CLASS)
print("Total examples:    ", N_POSTERIOR_CONFIG)
print("Sequence length:   ", SEQ_LEN)
print("Prior std:         ", PRIOR_STD)
print("Warmup steps:      ", WARMUP_STEPS)
print("Samples:           ", N_SAMPLES)
print("Chains:            ", N_CHAINS)
print("Thinning:          ", N_THINNING)
print("Target ESS:        ", NUM_EFFECTIVE_SAMPLES)
print("Step size init:    ", STEP_SIZE_INIT)
print("Eval batch size:   ", EVAL_BATCH_SIZE)

print()
print("Warmup phases:")
print("Phase 1:", int(WARMUP_STEPS * 0.8))
print("Phase 2:", int(WARMUP_STEPS * 0.1))
print("Phase 3:", int(WARMUP_STEPS * 0.1))

print("======================================")


In [ ]:
# ============================================================
# IMPORTS AND ENVIRONMENT
# ============================================================

import os
import sys
import gc
import copy
import json
import runpy
import importlib

import numpy as np
import jax
import jax.numpy as jnp

from jax import random
from jax.flatten_util import ravel_pytree


RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

os.chdir(QWEN_REPO)

if str(MILE_REPO) not in sys.path:
    sys.path.insert(
        0,
        str(MILE_REPO),
    )

if str(QWEN_REPO) not in sys.path:
    sys.path.insert(
        0,
        str(QWEN_REPO),
    )


print("Python:", sys.executable)
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

print("\nPaths:")
print("Qwen repo:", QWEN_REPO.exists())
print("MILE repo:", MILE_REPO.exists())
print("Baseline script:", BASELINE_SCRIPT.exists())
print("Result directory:", RESULT_DIR)

assert QWEN_REPO.exists()
assert MILE_REPO.exists()
assert BASELINE_SCRIPT.exists()


### FIX: `_predict` lax.cond bypass (jetzt aus `qwen_posterior_utils`)

`SubspaceBaseModel._predict` routet immer ueber `jax.lax.cond(train, train_fn, eval_fn, inputs)`, auch wenn `train` ein konkreter Python-Bool ist. `train_fn` ruft das echte Qwen-Modell mit `train=True` auf (ein Pfad, den wir fuer Posterior-Sampling nie brauchen/validieren). Unter `jax.lax.scan`-Tracing verliert `train` seinen konkreten Status, JAX baut/differenziert dann auch den ungenutzten `train_fn`-Zweig -- ein dort erzeugtes inf kann ueber `lax.cond`s Backward-Regel in den Gradienten des eigentlich gewaehlten `eval_fn`-Zweigs durchsickern.

Der Patch liegt jetzt in `qwen_posterior_utils.patch_subspace_curve_predict()` (geteilt mit MFVI/Laplace), hier nur noch der Aufruf.


In [ ]:
# ============================================================
# GETEILTES MODUL
# ============================================================

import qwen_posterior_utils as qpu

qpu.patch_subspace_curve_predict()


In [ ]:
# ============================================================
# LOAD QWEN AG-NEWS BASELINE
# ============================================================

baseline_objects = runpy.run_path(
    str(BASELINE_SCRIPT)
)

env = baseline_objects["env"]
params = baseline_objects["params"]
data = baseline_objects["data"]
rng_key = baseline_objects["rng_key"]

val_metrics = baseline_objects["val_metrics"]
test_metrics = baseline_objects["test_metrics"]


print("\nBaseline loaded")
print("Model:", type(env.s_model))
print("Parameter keys:", params.keys())

print("\nBaseline test metrics:")

for key, value in test_metrics.items():
    print(
        f"{key}: {float(value):.6f}"
    )


### Posterior-Setup (ersetzt `BALANCED SUBSET`, `MEMORY-EFFICIENT NLL`, `VECTORIZE`, `REBUILD`, `LOG POSTERIOR` -- jetzt alles in `qwen_posterior_utils`)


In [ ]:
# ============================================================
# POSTERIOR SETUP (Subset, NLL, Vektorisierung, Log-Posterior)
# ============================================================

posterior = qpu.setup_qwen_posterior(
    env=env,
    params=params,
    data=data,
    n_per_class=N_PER_CLASS,
    seq_len=SEQ_LEN,
    posterior_key_seed=POSTERIOR_KEY_SEED,
    prior_std=PRIOR_STD,
)

x_posterior = posterior.x_posterior
y_posterior = posterior.y_posterior
posterior_example_keys = posterior.posterior_example_keys
N_POSTERIOR_EXAMPLES = posterior.n_posterior_examples
subset_indices_host = posterior.subset_indices_host

theta_map = posterior.theta_map
rebuild_full_params = posterior.rebuild_full_params
qwen_log_posterior = posterior.qwen_log_posterior


In [ ]:
# ============================================================
# FULL POSTERIOR GRADIENT CHECK
# ============================================================

print("======================================")
print("FULL POSTERIOR GRADIENT CHECK")
print("======================================")

print(
    "Posterior examples:",
    N_POSTERIOR_EXAMPLES,
)

print(
    "Theta shape:",
    theta_map.shape,
)

print(
    "Theta dtype:",
    theta_map.dtype,
)


print()
print("Computing full posterior gradient...")


log_post_value, log_post_gradient = (
    jax.value_and_grad(
        qwen_log_posterior
    )(
        theta_map
    )
)

log_post_value = jax.block_until_ready(log_post_value)
log_post_gradient = jax.block_until_ready(log_post_gradient)

log_post_value_host = float(jax.device_get(log_post_value))
gradient_host = np.asarray(jax.device_get(log_post_gradient))

log_post_finite = bool(np.isfinite(log_post_value_host))
gradient_finite = bool(np.isfinite(gradient_host).all())
gradient_nan_count = int(np.isnan(gradient_host).sum())
gradient_inf_count = int(np.isinf(gradient_host).sum())
gradient_norm = float(np.linalg.norm(gradient_host))
gradient_max_abs = float(np.max(np.abs(gradient_host)))

print()
print("======================================")
print("RESULT")
print("======================================")
print("Log posterior:", log_post_value_host)
print("Log posterior finite:", log_post_finite)
print()
print("Gradient finite:", gradient_finite)
print("Gradient NaNs:", gradient_nan_count)
print("Gradient Infs:", gradient_inf_count)
print("Gradient norm:", gradient_norm)
print("Gradient max abs:", gradient_max_abs)
print("======================================")

assert log_post_finite, "Log posterior is not finite."
assert gradient_finite, "Full posterior gradient contains NaN or Inf."

print()
print("SUCCESS: Full posterior and gradient are finite.")


### Optional: manuelle MCLMC-Schritte (fixe step_size, keine Adaption)

Nicht zwingend noetig -- nur zum schnellen Vorab-Check, falls die automatische Warmup-Adaption unten wieder divergiert. Kann uebersprungen werden.


In [ ]:
# ============================================================
# MANUAL MCLMC DIAGNOSTIC (umgeht automatische Warmup-Adaption)
# ============================================================

import blackjax

jax.config.update("jax_debug_nans", False)

TEST_STEP_SIZES = [1e-5, 1e-6, 1e-7, 1e-8]
L_FIXED = 1.0
N_STEPS = 20

rng_key_diag = random.PRNGKey(0)

kernel = blackjax.mcmc.mclmc.build_kernel(
    logdensity_fn=qwen_log_posterior,
    integrator=blackjax.mcmc.integrators.isokinetic_mclachlan,
    sqrt_diag_cov=jnp.ones_like(theta_map),
)

for step_size in TEST_STEP_SIZES:
    print(f"\n=== step_size={step_size} ===")

    rng_key_diag, init_key = random.split(rng_key_diag)
    state = blackjax.mcmc.mclmc.init(
        position=theta_map,
        logdensity_fn=qwen_log_posterior,
        rng_key=init_key,
    )

    diverged_at = None

    for i in range(N_STEPS):
        rng_key_diag, step_key = random.split(rng_key_diag)

        state, info = kernel(step_key, state, L_FIXED, step_size)

        pos_norm = float(jnp.linalg.norm(state.position))
        ld = float(state.logdensity)
        finite = bool(jnp.isfinite(ld)) and bool(jnp.isfinite(pos_norm))

        if i < 5 or not finite:
            print(
                f"step {i:3d}  logdensity={ld:14.3f}  "
                f"|position|={pos_norm:14.3f}  finite={finite}"
            )

        if not finite:
            diverged_at = i
            break

    if diverged_at is None:
        print(f"STABIL ueber {N_STEPS} Schritte bei step_size={step_size}")
    else:
        print(f"DIVERGIERT bei Schritt {diverged_at}, step_size={step_size}")


### Fixe MCLMC-Konfiguration (automatische Adaption zweimal divergiert)

Die automatische Schrittweiten-/L-Adaption (`warmup_mclmc`) ist in zwei Versuchen (inkl. konservativerer Config) auf NaN divergiert -- im zweiten Versuch wuchs die getunte `step_size` sogar auf `0.06`, statt kleiner zu werden. Statt einem dritten Rateversuch an den Adaptions-Parametern nutzen wir die feste Konfiguration (`step_size=STEP_SIZE_INIT`, `L=1.0`), die in der Diagnose-Zelle oben bereits ueber 20 Schritte nachweislich stabil war -- jetzt aber ueber `jax.lax.scan` statt eager Loop, also deutlich schneller.


In [ ]:
# ============================================================
# FIXED-STEP MCLMC (keine automatische Adaption)
# ============================================================

import blackjax

FIXED_STEP_SIZE = STEP_SIZE_INIT
FIXED_L = 1.0

print("Fixed step_size:", FIXED_STEP_SIZE)
print("Fixed L:         ", FIXED_L)

mclmc_kernel = blackjax.mcmc.mclmc.build_kernel(
    logdensity_fn=qwen_log_posterior,
    integrator=blackjax.mcmc.integrators.isokinetic_mclachlan,
    sqrt_diag_cov=jnp.ones_like(theta_map),
)


def one_mclmc_step(state, key):
    new_state, info = mclmc_kernel(
        key, state, FIXED_L, FIXED_STEP_SIZE,
    )
    return new_state, new_state.position


rng_key, init_key = random.split(rng_key)

initial_state = blackjax.mcmc.mclmc.init(
    position=theta_map,
    logdensity_fn=qwen_log_posterior,
    rng_key=init_key,
)


In [ ]:
# ============================================================
# BURN-IN (WARMUP_STEPS Schritte, feste step_size, kein Tuning)
# ============================================================

rng_key, burnin_key = random.split(rng_key)
burnin_keys = random.split(burnin_key, WARMUP_STEPS)

warmup_state, _ = jax.lax.scan(
    one_mclmc_step,
    initial_state,
    burnin_keys,
)

warmup_state = jax.block_until_ready(warmup_state)

warmup_position_host = np.asarray(jax.device_get(warmup_state.position))
warmup_is_finite = bool(np.isfinite(warmup_position_host).all())

print("Burn-in completed")
print("Warmup position finite:", warmup_is_finite)
print("NaN count:", int(np.isnan(warmup_position_host).sum()))
print("Inf count:", int(np.isinf(warmup_position_host).sum()))

assert warmup_is_finite, (
    "Fixed-step burn-in still produced NaN/Inf -- step_size is too "
    "large even without adaptation. Try a smaller STEP_SIZE_INIT."
)


In [ ]:
# ============================================================
# MCLMC SAMPLING (N_SAMPLES Schritte, gleiche feste Konfiguration)
# ============================================================

rng_key, sampling_key = random.split(rng_key)
sampling_keys = random.split(sampling_key, N_SAMPLES)

final_state, sample_positions = jax.lax.scan(
    one_mclmc_step,
    warmup_state,
    sampling_keys,
)

final_state = jax.block_until_ready(final_state)
sample_positions = jax.block_until_ready(sample_positions)

samples_host = np.asarray(jax.device_get(sample_positions))

finite_per_sample = np.isfinite(samples_host).all(axis=1)

print("Sampling completed")
print("Samples shape:", samples_host.shape)
print("Finite per sample:", finite_per_sample)
print("All samples finite:", finite_per_sample.all())

assert finite_per_sample.all(), (
    "At least one sample contains NaN or Inf."
)


In [ ]:
# ============================================================
# BASIC SAMPLE DIAGNOSTICS
# ============================================================

distances_from_map = qpu.compute_distances_from_map(
    theta_map,
    sample_positions,
)


In [ ]:
# ============================================================
# EVALUATION DATA (Posterior-Subset oder echtes Testset)
# ============================================================

if EVAL_ON_TEST_SET:
    x_eval, y_eval = data.get("test")
    print("Evaluating on the full AG News TEST set.")
else:
    x_eval, y_eval = x_posterior, y_posterior
    print("Evaluating on the posterior subset (dev/stability check).")

print("Evaluation examples:", int(y_eval.shape[0]))


In [ ]:
# ============================================================
# POSTERIOR PREDICTIVE PROBABILITIES
# ============================================================

sample_probabilities, mean_probabilities, rng_key = (
    qpu.compute_posterior_predictive_probabilities(
        env=env,
        rebuild_full_params=rebuild_full_params,
        sample_thetas=samples_host,
        x_eval=x_eval,
        y_eval=y_eval,
        rng_key=rng_key,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
)


In [ ]:
# ============================================================
# METRICS
# ============================================================

metrics = qpu.compute_predictive_metrics(
    sample_probabilities=sample_probabilities,
    mean_probabilities=mean_probabilities,
    y_true=y_eval,
)

accuracy = metrics["accuracy"]
lppd = metrics["lppd"]
posterior_predictive_nll = metrics["posterior_predictive_nll"]
brier_score = metrics["brier_score"]
predictive_entropy = metrics["predictive_entropy"]
expected_entropy = metrics["expected_entropy"]
mutual_information = metrics["mutual_information"]


In [ ]:
# ============================================================
# EXPECTED CALIBRATION ERROR
# ============================================================

ece = qpu.multiclass_ece(
    mean_probabilities,
    y_eval,
    n_bins=15,
)

print("ECE:", float(ece))


In [ ]:
# ============================================================
# LOG POSTERIOR VALUES FOR ALL SAMPLES
# ============================================================

sample_log_posteriors_host = qpu.compute_sample_log_posteriors(
    qwen_log_posterior,
    sample_positions,
)


In [ ]:
# ============================================================
# SAVE RUN
# ============================================================

summary, metadata_arrays = qpu.summarize_metrics(metrics)

summary["ece"] = float(ece)

tuned_step_size = FIXED_STEP_SIZE
tuned_L = FIXED_L

summary.update({
    "run_name": RUN_NAME,
    "method": "MILE",
    "sampler": "MCLMC",
    "dataset": "AG News",
    "model": "Qwen2.5-0.5B",

    "seed": SEED,
    "posterior_key_seed": POSTERIOR_KEY_SEED,

    "n_per_class": N_PER_CLASS,
    "n_posterior_examples": N_POSTERIOR_EXAMPLES,
    "sequence_length": SEQ_LEN,

    "eval_on_test_set": EVAL_ON_TEST_SET,
    "n_eval_examples": int(y_eval.shape[0]),

    "prior_std": PRIOR_STD,

    "n_chains": N_CHAINS,
    "warmup_steps": WARMUP_STEPS,
    "n_samples": N_SAMPLES,
    "n_thinning": N_THINNING,
    "diagonal_preconditioning": DIAGONAL_PRECONDITIONING,
    "desired_energy_var_start": DESIRED_ENERGY_VAR_START,
    "desired_energy_var_end": DESIRED_ENERGY_VAR_END,
    "trust_in_estimate": TRUST_IN_ESTIMATE,
    "num_effective_samples": NUM_EFFECTIVE_SAMPLES,
    "step_size_init": STEP_SIZE_INIT,
    "adaptive_warmup": False,

    "tuned_step_size": tuned_step_size,
    "tuned_L": tuned_L,

    "all_samples_finite": bool(np.isfinite(samples_host).all()),
    "warmup_position_finite": bool(np.isfinite(warmup_position_host).all()),
    "n_parameter_dimensions": int(samples_host.shape[-1]),

    "minimum_distance_from_map": float(distances_from_map.min()),
    "mean_distance_from_map": float(distances_from_map.mean()),
    "maximum_distance_from_map": float(distances_from_map.max()),

    "minimum_log_posterior": float(sample_log_posteriors_host.min()),
    "mean_log_posterior": float(sample_log_posteriors_host.mean()),
    "maximum_log_posterior": float(sample_log_posteriors_host.max()),
})

metadata_arrays.update({
    "subset_indices": np.asarray(subset_indices_host),
    "labels": np.asarray(jax.device_get(y_eval)),
    "distances_from_map": np.asarray(distances_from_map),
    "log_posteriors": sample_log_posteriors_host,
})

paths = qpu.save_method_run(
    result_dir=RESULT_DIR,
    run_name=RUN_NAME,
    sample_positions=sample_positions,
    sample_probabilities=sample_probabilities,
    metadata_arrays=metadata_arrays,
    summary=summary,
)

print()
print("Results:")
print("Accuracy:           ", summary["accuracy"])
print("LPPD:               ", summary["lppd"])
print("NLL:                ", summary["posterior_predictive_nll"])
print("Brier Score:        ", summary["brier_score"])
print("ECE:                ", summary["ece"])
print("Mean pred. entropy: ", summary["mean_predictive_entropy"])
print("Mean exp. entropy:  ", summary["mean_expected_entropy"])
print("Mean MI:            ", summary["mean_mutual_information"])
